## 1. Verificar GPU Disponível

In [ ]:
!nvidia-smi

## 2. Instalar Dependências

In [ ]:
!pip install -q diffusers transformers accelerate torch pillow

## 3. Implementação da Classe KandinskyInpainting

Replica exatamente o código de `kandinsky_inpainting.py`

In [ ]:
from PIL import Image
import torch
from diffusers import KandinskyV22InpaintPipeline, KandinskyV22PriorPipeline

class KandinskyInpainting:
    def __init__(self, 
                 model_id: str = "kandinsky-community/kandinsky-2-2-decoder-inpaint",
                 prior_id: str = "kandinsky-community/kandinsky-2-2-prior",
                 device: str = None,
                 num_inference_steps: int = 50,
                 guidance_scale: float = 4.0):
        self.model_id = model_id
        self.prior_id = prior_id
        self.device = device if device else ("cuda" if torch.cuda.is_available() else "cpu")
        self.num_inference_steps = num_inference_steps
        self.guidance_scale = guidance_scale
        self.is_loaded = False
        self.model = None
        self.prior = None
        
        print(f"✓ Inicializado KandinskyInpainting")
        print(f"  Device: {self.device}")
        print(f"  Inference steps: {self.num_inference_steps}")
        print(f"  Guidance scale: {self.guidance_scale}")

    def load_model(self):
        """Carrega os modelos (prior e inpainting pipeline)"""
        if self.is_loaded:
            print("✓ Modelos já carregados")
            return
        
        print("Carregando Kandinsky Prior...")
        self.prior = KandinskyV22PriorPipeline.from_pretrained(
            self.prior_id,
            torch_dtype=torch.float16 if self.device == "cuda" else torch.float32
        )
        self.prior = self.prior.to(self.device)
        print("✓ Prior carregado")
        
        print("Carregando Kandinsky Inpaint Pipeline...")
        self.model = KandinskyV22InpaintPipeline.from_pretrained(
            self.model_id,
            torch_dtype=torch.float16 if self.device == "cuda" else torch.float32
        )
        self.model = self.model.to(self.device)
        self.is_loaded = True
        print("✓ Pipeline de inpainting carregado")

    def inpaint(self, image: Image.Image, mask: Image.Image, **kwargs) -> Image.Image:
        """Executa inpainting na imagem usando a máscara"""
        if not self.is_loaded:
            self.load_model()
        
        # Detecção automática do tipo de imagem baseado no nome do arquivo (se fornecido)
        image_path = str(kwargs.get("image_path", "")).lower()
        
        # Detectar tipo e definir prompt apropriado
        if 'antiga' in image_path or 'vintage' in image_path or 'old' in image_path or 'restored' in image_path:
            # Prompt otimizado para fotos antigas - foco em PRESERVAÇÃO FIEL
            default_prompt = (
                "professional photo restoration, complete only the missing areas, "
                "preserve existing facial features exactly as they are, "
                "match the vintage sepia tone perfectly, "
                "natural continuation of visible skin texture, "
                "period-appropriate clothing style from visible context, "
                "seamless repair maintaining original composition, "
                "no added objects, no decorations, no flowers, "
                "invisible restoration, faithful reproduction, "
                "match grain and lighting of surrounding area"
            )
            default_guidance = 7.0  # Guidance mais alto para seguir o prompt fielmente
            default_steps = 75  # Mais steps para melhor qualidade
        elif 'baixa_luz' in image_path or 'low_light' in image_path or 'noturna' in image_path:
            default_prompt = (
                "low light photography, dim lighting, natural shadows, "
                "ambient darkness, subtle illumination, night scene, "
                "photographic grain, authentic low light atmosphere, "
                "preserve existing lighting conditions, natural continuation"
            )
            default_guidance = 3.5
            default_steps = 50
        elif 'satelite' in image_path or 'satellite' in image_path or 'aerial' in image_path:
            default_prompt = (
                "aerial satellite imagery, top-down view, earth from above, "
                "natural terrain, vegetation patterns, land formations, "
                "geographic features, consistent satellite perspective, "
                "seamless terrain continuation, natural landscape"
            )
            default_guidance = 4.5
            default_steps = 50
        else:
            # Genérico - para uso quando não há informação sobre o tipo
            default_prompt = (
                "natural continuation, seamless completion, "
                "consistent style, coherent image, matching context, "
                "preserve composition, no added objects"
            )
            default_guidance = 4.0
            default_steps = 50
        
        prompt = kwargs.get("prompt", default_prompt)
        guidance_scale = kwargs.get("guidance_scale", default_guidance)
        num_inference_steps = kwargs.get("num_inference_steps", default_steps)
        
        # Negative prompt mais agressivo para evitar adições indesejadas
        negative_prompt = kwargs.get(
            "negative_prompt", 
            "added objects, extra items, flowers, decorations, jewelry, accessories, "
            "creative additions, new elements, fictional content, "
            "low quality, blurry, distorted, artifacts, inconsistent, "
            "watermark, text, logo, signature, unrealistic, "
            "obvious boundaries, visible seams, different style, different lighting"
        )
        
        # Kandinsky requer múltiplos de 64
        w, h = image.size
        new_w = (w // 64) * 64
        new_h = (h // 64) * 64
        
        # Garantir tamanho mínimo de 512
        if new_w < 512:
            new_w = 512
        if new_h < 512:
            new_h = 512
        
        if (w, h) != (new_w, new_h):
            print(f"Redimensionando de {w}x{h} para {new_w}x{new_h}")
            image = image.resize((new_w, new_h), Image.Resampling.LANCZOS)
            mask = mask.resize((new_w, new_h), Image.Resampling.NEAREST)
        
        print(f"\nParâmetros:")
        print(f"  guidance_scale: {guidance_scale}")
        print(f"  num_inference_steps: {num_inference_steps}")
        print(f"  prompt: {prompt[:100]}...")
        print(f"\nGerando embeddings do prior...")
        
        image_embeds, negative_image_embeds = self.prior(
            prompt=prompt,
            negative_prompt=negative_prompt
        ).to_tuple()
        
        print("Executando inpainting...")
        result = self.model(
            image=image,
            mask_image=mask,
            image_embeds=image_embeds,
            negative_image_embeds=negative_image_embeds,
            num_inference_steps=num_inference_steps,
            guidance_scale=guidance_scale
        ).images[0]
        
        print("✓ Inpainting concluído")
        return result

    def unload_model(self):
        """Libera memória da GPU"""
        if self.model is not None:
            del self.model
            self.model = None
        
        if self.prior is not None:
            del self.prior
            self.prior = None
        
        self.is_loaded = False
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        print("✓ Modelos removidos da memória")

## 4. Upload de Imagens

Faça upload da imagem original e da máscara (áreas brancas serão preenchidas)

In [ ]:
from google.colab import files
from IPython.display import display
import io

print("Upload da IMAGEM ORIGINAL:")
uploaded_image = files.upload()
image_filename = list(uploaded_image.keys())[0]
image = Image.open(io.BytesIO(uploaded_image[image_filename]))

print("\nUpload da MÁSCARA (áreas brancas = preencher):")
uploaded_mask = files.upload()
mask_filename = list(uploaded_mask.keys())[0]
mask = Image.open(io.BytesIO(uploaded_mask[mask_filename]))

print(f"\n✓ Imagem carregada: {image.size}")
print(f"✓ Máscara carregada: {mask.size}")

# Visualizar inputs
print("\nImagem original:")
display(image.resize((400, int(400 * image.size[1] / image.size[0]))))
print("\nMáscara:")
display(mask.resize((400, int(400 * mask.size[1] / mask.size[0]))))

## 5. Executar Inpainting com Parâmetros Otimizados

### Opção A: Detecção Automática (Recomendado para fotos antigas)

In [ ]:
# Criar modelo
model = KandinskyInpainting()

# Executar com detecção automática
# Se o nome do arquivo contém 'antiga', 'vintage', 'old' ou 'restored',
# os parâmetros otimizados serão aplicados automaticamente
result = model.inpaint(image, mask, image_path=image_filename)

# Visualizar resultado
print("\n" + "="*50)
print("RESULTADO:")
print("="*50)
display(result.resize((600, int(600 * result.size[1] / result.size[0]))))

# Salvar resultado
result.save("resultado_kandinsky.png")
print("\n✓ Resultado salvo como 'resultado_kandinsky.png'")

### Opção B: Parâmetros Customizados

Para controle total dos parâmetros:

In [ ]:
# Criar modelo (se ainda não criado)
if 'model' not in globals():
    model = KandinskyInpainting()

# Definir parâmetros customizados
custom_params = {
    "prompt": (
        "professional photo restoration, complete only the missing areas, "
        "preserve existing facial features exactly as they are, "
        "no added objects, no decorations, no flowers, "
        "faithful reproduction"
    ),
    "negative_prompt": (
        "flowers, decorations, jewelry, added objects, extra items, "
        "creative additions, new elements"
    ),
    "guidance_scale": 8.0,  # Valores mais altos = mais fiel ao prompt
    "num_inference_steps": 100  # Mais steps = melhor qualidade
}

# Executar
result_custom = model.inpaint(image, mask, **custom_params)

# Visualizar
print("\n" + "="*50)
print("RESULTADO (Parâmetros Customizados):")
print("="*50)
display(result_custom.resize((600, int(600 * result_custom.size[1] / result_custom.size[0]))))

# Salvar
result_custom.save("resultado_kandinsky_custom.png")
print("\n✓ Resultado salvo como 'resultado_kandinsky_custom.png'")

## 6. Comparação Lado a Lado

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

axes[0].imshow(image)
axes[0].set_title('Imagem Original', fontsize=14)
axes[0].axis('off')

axes[1].imshow(mask, cmap='gray')
axes[1].set_title('Máscara', fontsize=14)
axes[1].axis('off')

axes[2].imshow(result)
axes[2].set_title('Resultado Kandinsky', fontsize=14)
axes[2].axis('off')

plt.tight_layout()
plt.show()

## 7. Download dos Resultados

In [ ]:
from google.colab import files

# Baixar resultado
files.download('resultado_kandinsky.png')

# Se você executou a versão customizada, baixar também
try:
    files.download('resultado_kandinsky_custom.png')
except:
    pass

## 8. Liberar Memória (Opcional)

In [ ]:
# Liberar GPU
model.unload_model()

# Verificar memória da GPU
!nvidia-smi

---

## 📝 Notas sobre Parâmetros

### `guidance_scale` (Controle de Fidelidade ao Prompt)
- **Valores baixos (1-3):** Mais criativo, pode ignorar o prompt e adicionar elementos aleatórios
- **Valores médios (4-6):** Balanceado
- **Valores altos (7-10):** Segue o prompt fielmente, menos criativo ✅ **Recomendado para fotos antigas**

### `num_inference_steps` (Qualidade)
- **20-40:** Rápido, menor qualidade
- **50-75:** Bom balanço ✅ **Padrão**
- **75-150:** Melhor qualidade, mais lento ✅ **Recomendado para restauração**

### Detecção Automática de Tipo
O modelo detecta automaticamente o tipo de imagem pelo nome do arquivo:
- **Fotos antigas:** Nomes contendo `antiga`, `vintage`, `old`, `restored`
  - guidance_scale: 7.0
  - num_inference_steps: 75
  - Prompt otimizado para preservação fiel
  
- **Baixa luz:** Nomes contendo `baixa_luz`, `low_light`, `noturna`
  - guidance_scale: 3.5
  - num_inference_steps: 50
  
- **Satélite:** Nomes contendo `satelite`, `satellite`, `aerial`
  - guidance_scale: 4.5
  - num_inference_steps: 50

### Troubleshooting

**Problema:** Modelo adiciona flores/objetos aleatórios
- ✅ Aumentar `guidance_scale` para 8.0-9.0
- ✅ Aumentar `num_inference_steps` para 100
- ✅ Adicionar ao `negative_prompt`: "flowers, decorations, jewelry"

**Problema:** Resultado borrado
- ✅ Aumentar `num_inference_steps` para 75-100

**Problema:** Muito lento
- ✅ Reduzir `num_inference_steps` para 50 (sacrifica qualidade)

---

## 🎯 Exemplo de Uso para Foto Antiga

```python
# Parâmetros otimizados para evitar adições indesejadas
result = model.inpaint(
    image, 
    mask,
    prompt="complete the face naturally, no flowers, no jewelry",
    negative_prompt="flowers, decorations, jewelry, added objects",
    guidance_scale=8.0,  # Alto = mais fiel ao prompt
    num_inference_steps=100  # Máxima qualidade
)
```